In [ ]:
!pip install datasets

In [ ]:
import os
import json
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

TRAIN_DIR = "/kaggle/working/plan_b_train_data"
TARGET_SIZE = 1024
os.makedirs(TRAIN_DIR, exist_ok=True)

# 1. Download ONLY the first 50 images for quick testing
print("Downloading FIRST 50 images from pixel-art-nouns-2k for test run...")
dataset = load_dataset("jiovine/pixel-art-nouns-2k", split="train[:50]")   # ← ONLY CHANGE HERE

metadata = []
print(f"Upscaling {len(dataset)} images using Nearest Neighbor...")

for i, item in enumerate(tqdm(dataset)):
    try:
        img = item['image']
        
        # Keep your style trigger (good for pixel art)
        caption = "pixel art style, " + item['text'] 
        
        if img.mode != "RGB":
            img = img.convert("RGB")
            
        # Nearest Neighbor upscaling
        img_upscaled = img.resize((TARGET_SIZE, TARGET_SIZE), resample=Image.NEAREST)
        
        filename = f"noun_{i}.png"
        img_upscaled.save(os.path.join(TRAIN_DIR, filename), "PNG")
        
        metadata.append({"file_name": filename, "text": caption})
        
    except Exception as e:
        print(f"Skipping image {i} due to error: {e}")

# 3. Save the JSONL file for the trainer
with open(os.path.join(TRAIN_DIR, "metadata.jsonl"), 'w') as f:
    for entry in metadata:
        f.write(json.dumps(entry) + "\n")
        
print(f"✅ Test data ready! Only {len(dataset)} images saved to {TRAIN_DIR}")

In [ ]:
# import os
# import json
# from datasets import load_dataset
# from PIL import Image
# from tqdm import tqdm

# TRAIN_DIR = "/kaggle/working/plan_b_train_data"
# TARGET_SIZE = 1024  # Upscale target for SSD-1B
# os.makedirs(TRAIN_DIR, exist_ok=True)

# # 1. Download directly from Hugging Face
# print("Downloading pixel-art-nouns-2k from Hugging Face...")
# dataset = load_dataset("jiovine/pixel-art-nouns-2k", split="train")

# metadata = []
# print(f"Upscaling {len(dataset)} images using Nearest Neighbor...")

# for i, item in enumerate(tqdm(dataset)):
#     try:
# #!pip install datasets        # HF datasets automatically load images as PIL objects
#         img = item['image']
        
#         # The text column in this specific dataset is usually 'text'
#         # Add the mandatory style trigger here to prevent catastrophic forgetting
#         caption = "pixel art style, " + item['text'] 
        
#         if img.mode != "RGB":
#             img = img.convert("RGB")
            
#         # 2. Crucial Step: Nearest Neighbor upscaling to keep pixels sharp
#         img_upscaled = img.resize((TARGET_SIZE, TARGET_SIZE), resample=Image.NEAREST)
        
#         filename = f"noun_{i}.png"
#         img_upscaled.save(os.path.join(TRAIN_DIR, filename), "PNG")
        
#         metadata.append({"file_name": filename, "text": caption})
        
#     except Exception as e:
#         print(f"Skipping image {i} due to error: {e}")

# # 3. Save the JSONL file for the trainer
# with open(os.path.join(TRAIN_DIR, "metadata.jsonl"), 'w') as f:
#     for entry in metadata:
#         f.write(json.dumps(entry) + "\n")
        
# print(f"Plan B Data Ready! Saved to {TRAIN_DIR}")

In [ ]:
# === PIXART-SIGMA PIPELINE SETUP (STABLE FOR T4 x2) ===
!pip install -U -q diffusers accelerate transformers peft bitsandbytes wandb

# Use the stable PixArt-alpha repo (same Sigma model, reliable LoRA saving)
!git clone https://github.com/PixArt-alpha/PixArt-alpha.git
%cd PixArt-alpha

# Install requirements
!pip install -r requirements.txt

print("✅ Stable PixArt-Sigma LoRA pipeline ready for T4 x2!")

In [ ]:
# # ============================================================
# # W&B Setup — load API key from Kaggle Secrets and login
# # Make sure you've added your W&B API key as a Kaggle Secret
# # with the label "WANDB" (Add-ons → Secrets in notebook sidebar)
# # ============================================================
# import os
# import wandb
# from kaggle_secrets import UserSecretsClient

# wandb_api_key = UserSecretsClient().get_secret("WANDB")
# wandb.login(key=wandb_api_key)

# # Set the W&B project name so the training script picks it up
# os.environ["WANDB_PROJECT"] = "Pixel Art"

# print("Logged into W&B successfully. Project: 'Pixel Art'")

In [ ]:
import os
import shlex
import subprocess
from collections import deque

import torch
from accelerate.utils import write_basic_config

write_basic_config()

TRAIN_DIR = "/kaggle/working/plan_b_train_data"
MODEL_ID = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
OUTPUT_DIR = "/kaggle/working/pixart_sigma_pixelart_lora_TEST"
SCRIPT_REL = "train_scripts/train_pixart_lora_hf.py"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- Preflight checks (fail early with clear messages) ----------
if not os.path.isdir(TRAIN_DIR):
    raise FileNotFoundError(
        f"Training data directory not found: {TRAIN_DIR}. "
        "Run the data-prep cell first."
    )

metadata_path = os.path.join(TRAIN_DIR, "metadata.jsonl")
if not os.path.isfile(metadata_path):
    raise FileNotFoundError(
        f"Missing metadata file: {metadata_path}. "
        "Your train_data_dir must contain metadata.jsonl for imagefolder loading."
    )

image_count = sum(
    1
    for fn in os.listdir(TRAIN_DIR)
    if fn.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
)
if image_count == 0:
    raise RuntimeError(f"No image files found in {TRAIN_DIR}.")

# Find training script location robustly (helps after Kaggle runtime restarts).
candidate_roots = [
    os.getcwd(),
    "/kaggle/working/PixArt-alpha",
    "/kaggle/working/PixArt-sigma",
    "/kaggle/working/PixArt-Sigma",
]
script_root = next(
    (root for root in candidate_roots if os.path.isfile(os.path.join(root, SCRIPT_REL))),
    None,
)
if script_root is None:
    raise FileNotFoundError(
        "Could not find train_scripts/train_pixart_lora_hf.py. "
        "Run the setup/clone cell first and ensure repo checkout succeeded."
    )

os.chdir(script_root)
print(f"Using repo root: {script_root}")
print(f"Found training script: {os.path.join(script_root, SCRIPT_REL)}")
print(f"Detected {image_count} training images and metadata.jsonl")

gpu_count = torch.cuda.device_count()
if gpu_count < 1:
    raise RuntimeError("No CUDA device visible. Enable GPU in Kaggle settings.")

num_processes = min(2, gpu_count)
if num_processes < 2:
    print(
        f"Warning: only {gpu_count} GPU detected. "
        f"Falling back to num_processes={num_processes}."
    )

# Keep args in a Python list so unsupported flags fail fast and visibly.
cmd = [
    "accelerate", "launch",
    "--num_processes", str(num_processes),
    "--mixed_precision=fp16",
    SCRIPT_REL,
    "--pretrained_model_name_or_path", MODEL_ID,
    "--train_data_dir", TRAIN_DIR,
    "--caption_column", "text",
    "--resolution", "1024",
    "--train_batch_size", "2",
    "--gradient_accumulation_steps", "4",
    "--max_train_steps", "50",
    "--learning_rate", "1e-5",
    "--lr_scheduler", "cosine",
    "--lr_warmup_steps", "50",
    "--output_dir", OUTPUT_DIR,
    "--checkpointing_steps", "10",
    "--validation_prompt", "a cute pixel art cat wearing sunglasses",
    "--validation_epochs", "5",
    "--mixed_precision", "fp16",
    "--use_8bit_adam",
    "--gradient_checkpointing",
    "--noise_offset", "0.1",
    "--rank", "16",
]

print("Running training command:")
print(" ".join(shlex.quote(c) for c in cmd))

# Stream logs live and keep tail for focused error reporting.
log_path = os.path.join(OUTPUT_DIR, "quick_test_train.log")
tail = deque(maxlen=200)

with open(log_path, "w", encoding="utf-8") as log_file:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="")
        log_file.write(line)
        tail.append(line.rstrip())

    proc.wait()

if proc.returncode != 0:
    joined = "\n".join(tail).lower()
    hints = []

    if "unrecognized arguments" in joined:
        hints.append("Unsupported CLI argument passed to training script.")
    if "no such file or directory" in joined or "can't open file" in joined:
        hints.append("Training script path or working directory is wrong.")
    if "out of memory" in joined or "cuda error" in joined:
        hints.append("GPU OOM/driver issue. Try batch_size=1 or num_processes=1.")
    if "metadata.jsonl" in joined and "not" in joined:
        hints.append("Dataset folder is missing/invalid metadata.jsonl.")

    hint_text = "\n".join(f"- {h}" for h in hints) if hints else "- Check first traceback line in log tail."
    tail_text = "\n".join(list(tail)[-40:])

    raise RuntimeError(
        f"Training failed with exit code {proc.returncode}.\n"
        f"Log file: {log_path}\n"
        f"Likely cause(s):\n{hint_text}\n\n"
        f"Last log lines:\n{tail_text}"
    )

print(f"Quick test finished. Artifacts should be in: {OUTPUT_DIR}")
print("Top-level output contents:", sorted(os.listdir(OUTPUT_DIR)))
print(f"Full training log saved at: {log_path}")

In [ ]:
import os
import re
import torch
from diffusers import Transformer2DModel, PixArtSigmaPipeline
from peft import PeftModel
from PIL import Image

OUTPUT_DIR = "/kaggle/working/pixart_sigma_pixelart_lora_TEST"
MODEL_ID = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
PIPELINE_ID = "PixArt-alpha/pixart_sigma_sdxlvae_T5_diffusers"

if not os.path.isdir(OUTPUT_DIR):
    raise FileNotFoundError(f"Output directory not found: {OUTPUT_DIR}")

def print_tree(root_dir, max_depth=3):
    for root, dirs, files in os.walk(root_dir):
        rel = os.path.relpath(root, root_dir)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "    " * depth
        folder_name = os.path.basename(root) if rel != "." else os.path.basename(root_dir)
        print(f"{indent}{folder_name}/")
        for f in sorted(files):
            print(f"{indent}    {f}")

def find_latest_checkpoint(root_dir):
    checkpoints = []
    for name in os.listdir(root_dir):
        match = re.fullmatch(r"checkpoint-(\\d+)", name)
        path = os.path.join(root_dir, name)
        if match and os.path.isdir(path):
            checkpoints.append((int(match.group(1)), path))

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0], reverse=True)
    return checkpoints[0][1]

def has_peft_adapter(folder):
    config_ok = os.path.exists(os.path.join(folder, "adapter_config.json"))
    weight_ok = any(
        os.path.exists(os.path.join(folder, name))
        for name in ("adapter_model.safetensors", "adapter_model.bin", "pytorch_lora_weights.safetensors")
    )
    return config_ok and weight_ok

print("Output directory tree (depth <= 3):")
print_tree(OUTPUT_DIR, max_depth=3)

candidate_dirs = [OUTPUT_DIR]
latest_checkpoint = find_latest_checkpoint(OUTPUT_DIR)
if latest_checkpoint is not None:
    candidate_dirs.append(latest_checkpoint)

lora_path = next((d for d in candidate_dirs if has_peft_adapter(d)), None)

if lora_path is None:
    raise FileNotFoundError(
        "No PEFT LoRA adapter was found.\n"
        "Expected files in OUTPUT_DIR or latest checkpoint:\n"
        "  - adapter_config.json\n"
        "  - adapter_model.safetensors (or adapter_model.bin)\n"
        "Training may have failed before save. Check the training cell logs for unrecognized arguments or other errors."
    )

print(f"Using LoRA adapter path: {lora_path}")

# Load base model
transformer = Transformer2DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    torch_dtype=torch.float16,
)

# Load trained LoRA adapter
transformer = PeftModel.from_pretrained(transformer, lora_path)

# Create pipeline
pipe = PixArtSigmaPipeline.from_pretrained(
    PIPELINE_ID,
    transformer=transformer,
    torch_dtype=torch.float16,
)
pipe.to("cuda")
pipe.enable_model_cpu_offload()

# Generate test image
image = pipe(
    prompt="a pixel art character with square black glasses and a hotdog-shaped head",
    negative_prompt="blur, low quality, realistic, photo",
    num_inference_steps=50,
    guidance_scale=4.5,
    height=1024,
    width=1024,
).images[0]

# Downscale to 256x256 while preserving hard edges
image = image.resize((256, 256), resample=Image.NEAREST)
image.save("pixel_art_test.png")
image

In [ ]:
# import os

# OUTPUT_DIR = "/kaggle/working/pixart_sigma_pixelart_lora_TEST"

# print("=== FULL DIRECTORY CONTENTS OF OUTPUT_DIR ===")
# for root, dirs, files in os.walk(OUTPUT_DIR):
#     level = root.replace(OUTPUT_DIR, '').count(os.sep)
#     indent = '    ' * level
#     print(f"{indent}📂 {os.path.basename(root)}/")
#     subindent = '    ' * (level + 1)
#     for f in sorted(files):
#         print(f"{subindent}📄 {f}")

# print("\n✅ Directory listing complete. Copy-paste the entire output here.")